In [ ]:
import os
from dotenv import load_dotenv
from scraper import fetch_website_contents

from IPython.display import Markdown, display
from google import genai

load_dotenv()

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

api_key = os.getenv("GEMINI_API_KEY")

if not api_key:
    print("No Gemini API key was found.")
elif api_key.strip() != api_key:
    print("Gemini API key has spaces at the beginning or end.")
else:
    print("Gemini API key found! ✅")

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)

message = "Hello, Gemini! This is my first ever message to you! Hi!"

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=message,
    config=types.GenerateContentConfig(
        thinking_config=types.ThinkingConfig(
            thinking_budget=0  
        )
    ),
)

text_output = "".join(
    part.text
    for part in response.candidates[0].content.parts
    if part.text and not getattr(part, "thought", False)
)

print(text_output)

## OK onwards with our first project

In [ ]:
# Let's try out this utility

ed = fetch_website_contents("https://github.com/jamilhossain1997")
print(ed)

In [ ]:
# Define our system prompt - you can experiment with this later, changing the last sentence to 'Respond in markdown in Spanish."

system_prompt = """
You are a helpful assistant that summarizes website content.
Provide a clear and concise summary.
"""



In [ ]:
# Define our user prompt

user_prompt_prefix = """
Please summarize the following website:

"""

## Messages

The API from OpenAI expects to receive messages in a particular structure.
Many of the other APIs share this structure:

```python
[
    {"role": "system", "content": "system message goes here"},
    {"role": "user", "content": "user message goes here"}
]
```
To give you a preview, the next 2 cells make a rather simple call - we won't stretch the mighty GPT (yet!)

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)

system_prompt = "You are a helpful assistant."
user_message = "What is 2 + 2?"

response = client.models.generate_content(
    model="gemini-3.5-flash",
    contents=user_message,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        thinking_config=types.ThinkingConfig(
            thinking_level=types.ThinkingLevel.MINIMAL
        )
    ),
)

print(response.text)

In [ ]:
# See how this function creates exactly the format above

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [ ]:
# Try this out, and then try for a few more websites

messages_for(ed)

In [ ]:
from google import genai
from google.genai import types

client = genai.Client(api_key=api_key)


system_prompt = """
You are an expert website analyst.
Summarize the website clearly and concisely.
Focus on the person's profile, skills, experience, projects, and important information.
"""

user_prompt_prefix = """
Please summarize the following website:

"""


def messages_for(website):
    return [
        {
            "role": "system",
            "content": system_prompt
        },
        {
            "role": "user",
            "content": user_prompt_prefix + website
        }
    ]


def to_gemini_contents(messages):
    system_prompt = None
    contents = []

    for m in messages:
        if m["role"] == "system":
            system_prompt = m["content"]
        else:
            role = "model" if m["role"] == "assistant" else "user"

            contents.append({
                "role": role,
                "parts": [
                    {"text": m["content"]}
                ]
            })

    return system_prompt, contents


def summarize(url):
    website = fetch_website_contents(url)

    system_prompt, contents = to_gemini_contents(
        messages_for(website)
    )

    response = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt
        ),
    )

    text_output = "".join(
        part.text
        for part in response.candidates[0].content.parts
        if getattr(part, "text", None)
        and not getattr(part, "thought", False)
    )

    return text_output

In [ ]:
summarize("https://github.com/jamilhossain1997")


In [10]:
# A function to display this nicely in the output, using markdown

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

    print(summary)

In [11]:
display_summary("https://github.com/jamilhossain1997")

Based on the provided text, here is a summary of the website:

### **Profile Overview**
* **Name:** Md. Jamil Hossain
* **GitHub Username:** [jamilhossain1997](https://github.com/jamilhossain1997)

### **Note on Content**
The provided text contains the standard GitHub navigation menus, platform features, and system alerts, but **cuts off** right as the profile overview begins. 

To provide a detailed summary of Md. Jamil Hossain's specific skills, professional experience, repositories, and projects, please provide the text or details from the main body of his GitHub profile page.

Based on the provided text, here is a summary of the website:

### **Profile Overview**
* **Name:** Md. Jamil Hossain
* **GitHub Username:** [jamilhossain1997](https://github.com/jamilhossain1997)

### **Note on Content**
The provided text contains the standard GitHub navigation menus, platform features, and system alerts, but **cuts off** right as the profile overview begins. 

To provide a detailed summary of Md. Jamil Hossain's specific skills, professional experience, repositories, and projects, please provide the text or details from the main body of his GitHub profile page.


# Let's try more websites

Note that this will only work on websites that can be scraped using this simplistic approach.

Websites that are rendered with Javascript, like React apps, won't show up. See the community-contributions folder for a Selenium implementation that gets around this. You'll need to read up on installing Selenium (ask ChatGPT!)

Also Websites protected with CloudFront (and similar) may give 403 errors - many thanks Andy J for pointing this out.

But many websites will work just fine!

In [ ]:
display_summary("https://cnn.com")

In [ ]:
display_summary("https://anthropic.com")

In [ ]:
# Step 1: Create your prompts

system_prompt = "something here"
user_prompt = """
    Lots of text
    Can be pasted here
"""

# Step 2: Make the messages list

messages = [] # fill this in

# Step 3: Call OpenAI
# response =

# Step 4: print the result
# print(